# Hospital Payments — Validate Dimensional Model
Run after `2_etl.py` to verify row counts, schema, and analytical queries.

In [ ]:
from google.cloud import bigquery
import configparser
import pandas as pd

config = configparser.ConfigParser()
config.read('../bq.cfg')
PROJECT = config['BIGQUERY']['project_id']
DATASET = config['BIGQUERY']['dataset']

client = bigquery.Client(project=PROJECT)

def q(sql):
    return client.query(sql.replace('{PROJECT}', PROJECT).replace('{DATASET}', DATASET)).to_dataframe()

## 1. Row Counts

In [ ]:
q("""
SELECT 'fact_claims' AS tbl, COUNT(*) AS rows FROM `{PROJECT}.{DATASET}.fact_claims`
UNION ALL SELECT 'dim_hospitals', COUNT(*) FROM `{PROJECT}.{DATASET}.dim_hospitals`
UNION ALL SELECT 'dim_patients',  COUNT(*) FROM `{PROJECT}.{DATASET}.dim_patients`
UNION ALL SELECT 'dim_diagnoses', COUNT(*) FROM `{PROJECT}.{DATASET}.dim_diagnoses`
UNION ALL SELECT 'dim_date',      COUNT(*) FROM `{PROJECT}.{DATASET}.dim_date`
""")

## 2. Avg Payment by State

In [ ]:
df = q("""
SELECT h.state, COUNT(f.claim_key) AS total_claims,
       ROUND(AVG(f.avg_payment), 2) AS avg_payment
FROM `{PROJECT}.{DATASET}.fact_claims` f
JOIN `{PROJECT}.{DATASET}.dim_hospitals` h ON f.hospital_key = h.hospital_key
GROUP BY h.state ORDER BY avg_payment DESC
""")
df

## 3. Readmission Rate by Diagnosis

In [ ]:
q("""
SELECT d.diagnosis_name,
       COUNT(*) AS total_claims,
       ROUND(AVG(f.readmission_flag) * 100, 1) AS readmission_rate_pct
FROM `{PROJECT}.{DATASET}.fact_claims` f
JOIN `{PROJECT}.{DATASET}.dim_diagnoses` d ON f.diagnosis_key = d.diagnosis_key
GROUP BY d.diagnosis_name
ORDER BY readmission_rate_pct DESC
""")

## 4. Payer Mix

In [ ]:
q("""
SELECT payer_type, COUNT(*) AS total_claims,
       ROUND(AVG(avg_payment), 2) AS avg_payment
FROM `{PROJECT}.{DATASET}.fact_claims`
GROUP BY payer_type ORDER BY avg_payment DESC
""")